In [ ]:


import os
import pandas as pd
data_folder = '../data/'
path = os.path.join(data_folder)

df = pd.DataFrame()
df2 = pd.DataFrame()
for file in os.listdir(path):
    if 'price_history' in file:
        df2 = pd.read_csv(data_folder + file)
        df = pd.concat([df, df2])
        

meta_path = '../data/all_products_meta.csv'
meta_df = pd.read_csv(meta_path)

df = df.merge(meta_df, on = 'asin')

KEEPA_EPOCH = pd.Timestamp('2011-01-01')
df['listedSince'] = pd.to_datetime(KEEPA_EPOCH + pd.to_timedelta(df['listedSince'], 'min'))
df['datetime'] = pd.to_datetime(df['datetime'])
df['days_since_launch'] = (df['datetime'] - df['listedSince']).dt.days
df = df[df['days_since_launch'] > 0]

df['median'] = df.groupby('asin')['NEW'].transform('median')
df = df[df['NEW'] < 3 * df['median']]
df = df[df['NEW'].notna()]

df = df.set_index('datetime', drop = True)
backup_df = df
df = df.groupby('asin').resample('W')['NEW'].min().reset_index()
df['NEW'] = df.groupby('asin')['NEW'].ffill()

df = df.merge(meta_df[['asin', 'brand', 'model', 'listedSince']], on='asin', how='left')
df['listedSince'] = pd.to_datetime(KEEPA_EPOCH + pd.to_timedelta(df['listedSince'], 'min'))
df['days_since_launch'] = (df['datetime'] - df['listedSince']).dt.days

import plotly.express as px
iphone_13_df = df[df['model'] == 'iPhone 13']
chart = px.line(iphone_13_df, x = 'days_since_launch', y = 'NEW')
chart.show()

In [ ]:
all_iphones_df = df[df['brand'] == 'Apple']
print(all_iphones_df.head())

all_iphones_line_chart = px.line(
    all_iphones_df,
    x = 'days_since_launch',
    y = 'NEW',
    color = 'model'
    )

all_iphones_line_chart.show()

In [93]:
all_iphones_df['month'] = all_iphones_df['datetime'].dt.to_period('M')

summary = all_iphones_df.groupby(['asin', 'month'])['NEW'].agg(
    min_price='min',
    max_price='max',
    first_price='first',
    last_price='last'
).reset_index()

# summary.head()

all_iphones_df['first_price'] = all_iphones_df.groupby('model')['NEW'].transform('first')
all_iphones_df['last_price'] = all_iphones_df.groupby('model')['NEW'].transform('last')
all_iphones_df['max_price'] = all_iphones_df.groupby('model')['NEW'].transform('max')
all_iphones_df['min_price'] = all_iphones_df.groupby('model')['NEW'].transform('min')
all_iphones_df['weekly_drop'] = all_iphones_df.groupby('model')['NEW'].diff(1).abs()


summary = all_iphones_df.groupby('model')[['first_price', 'last_price']].first()
summary['total_drop_pct'] = round((summary['first_price'] - summary['last_price']) / summary['first_price'] * 100, 1)
summary = summary.sort_values('total_drop_pct', ascending = False)
summary['max_weekly_drop'] = all_iphones_df.groupby('model')['weekly_drop'].max()
summary.head(7)



,first_price,last_price,total_drop_pct,max_weekly_drop
model,,,,
iPhone 12,779.97,173.00,77.8,103.99
iPhone 11,527.00,159.96,69.6,38.54
iPhone 13,729.99,248.90,65.9,230.01
iPhone 14,799.97,277.95,65.3,562.25
iPhone 15,779.99,378.71,51.4,80.53
iPhone 16,1350.00,670.93,50.3,100.05
iPhone 17,849.00,771.82,9.1,66.97
